In [1]:
# Step 1: Install required packages
!pip install -q pandas numpy scikit-learn spacy transformers beautifulsoup4 nltk datasets
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 56.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
# Step 2: Import libraries
import os
import re
import json
import pandas as pd
from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split
import spacy
import nltk
from nltk.tokenize import sent_tokenize
from transformers import AutoTokenizer

nltk.download("punkt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [3]:
# Step 3: Load spaCy model and BART tokenizer
nlp = spacy.load("en_core_web_sm")

MODEL_NAME = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
!unzip /content/legal_summarization-master.zip

Archive:  /content/legal_summarization-master.zip
9d203292eb188e13523366490863b0cc14c89017
   creating: legal_summarization-master/
  inflating: legal_summarization-master/LICENSE.md  
  inflating: legal_summarization-master/README.md  
  inflating: legal_summarization-master/all_v1.json  
  inflating: legal_summarization-master/tldrlegal_v1.json  
  inflating: legal_summarization-master/tosdr_annotated_v1.json  


In [7]:
# Step 4: Load Laura Manor legal_summarization dataset
DATA_PATH = "/content/legal_summarization-master/tosdr_annotated_v1.json"


with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

# Create dataframe
df = pd.DataFrame(data)

print("Original shape:", df.shape)
print("Original columns:", df.columns.tolist()[:10])

# Transpose because rows are fields and columns are examples
df = df.T.reset_index(drop=False)
df = df.rename(columns={"index": "uid"})

print("Transposed shape:", df.shape)
print("Transposed columns:", df.columns.tolist())
display(df.head())

Original shape: (12, 361)
Original columns: ['tosdr001', 'tosdr002', 'tosdr003', 'tosdr004', 'tosdr005', 'tosdr006', 'tosdr007', 'tosdr008', 'tosdr009', 'tosdr010']
Transposed shape: (361, 13)
Transposed columns: ['uid', 'case_code', 'case_text', 'doc', 'note', 'original_text', 'reference_summary', 'title_code', 'title_text', 'uid', 'urls', 'tldr_code', 'tldr_text']


,uid,case_code,case_text,doc,note,original_text,reference_summary,title_code,title_text,uid,urls,tldr_code,tldr_text
0,tosdr001,"1,s",This service does not track you,Privacy Policy,,search encrypt does not track search history i...,this service does not track you.,"2,s",no tracking,tosdr001,{'searchencrypt.com'},NaN,NaN
1,tosdr002,NaN,NaN,Privacy Policy,,we also provide you additional data control op...,you can request access and deletion of persona...,"1,s",you can request access and deletion of persona...,tosdr002,{'stackoverflow.com'},NaN,NaN
2,tosdr003,"2,s",The copyright license that users grant to the ...,Additional Terms of Service,,rvices you grant oath the following worldwide ...,the copyright license granted to yahoo for pho...,"3,d","Yahoo's copyright license for photos, graphics...",tosdr003,{'flickr.com'},1,The copyright license granted to Yahoo for pho...
3,tosdr004,NaN,NaN,Terms and Conditions,,we may change these terms and conditions to re...,if you are a subscriber jagex will treat the f...,"2,s",Terms may be changed any time at their discret...,tosdr004,{'jagex.com'},1,"If you are a subscriber, Jagex will treat the ..."
4,tosdr005,"1,s",The service uses your personal data to employ ...,Privacy Policy,,it also enables us to serve you advertising an...,the service uses your personal data to employ ...,"2,d",targeted third-party advertising,tosdr005,{'academia.edu'},NaN,NaN


In [8]:
print(df.columns.tolist())

['uid', 'case_code', 'case_text', 'doc', 'note', 'original_text', 'reference_summary', 'title_code', 'title_text', 'uid', 'urls', 'tldr_code', 'tldr_text']


In [9]:
# Step 5: Select the correct columns
TEXT_COL = "original_text"
SUMMARY_COL = "reference_summary"
DOC_COL = "doc"

df = df[[TEXT_COL, SUMMARY_COL, DOC_COL]].dropna().reset_index(drop=True)

print("Filtered dataset shape:", df.shape)
display(df.head())

Filtered dataset shape: (361, 3)


,original_text,reference_summary,doc
0,search encrypt does not track search history i...,this service does not track you.,Privacy Policy
1,we also provide you additional data control op...,you can request access and deletion of persona...,Privacy Policy
2,rvices you grant oath the following worldwide ...,the copyright license granted to yahoo for pho...,Additional Terms of Service
3,we may change these terms and conditions to re...,if you are a subscriber jagex will treat the f...,Terms and Conditions
4,it also enables us to serve you advertising an...,the service uses your personal data to employ ...,Privacy Policy


In [11]:
# Step 6: Clean raw text
import re
from bs4 import BeautifulSoup

def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = BeautifulSoup(text, "html.parser").get_text(" ")
    text = text.replace("\u2019", "'").replace("\u201c", '"').replace("\u201d", '"')
    text = text.replace("\u2013", "-").replace("\u2014", "-")
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df[TEXT_COL].apply(clean_text)
df["clean_summary"] = df[SUMMARY_COL].apply(clean_text)

# Full-document version for extractive summarisation
extractive_df = df[[DOC_COL, "clean_text", "clean_summary"]].copy()

display(extractive_df.head())

,doc,clean_text,clean_summary
0,Privacy Policy,search encrypt does not track search history i...,this service does not track you.
1,Privacy Policy,we also provide you additional data control op...,you can request access and deletion of persona...
2,Additional Terms of Service,rvices you grant oath the following worldwide ...,the copyright license granted to yahoo for pho...
3,Terms and Conditions,we may change these terms and conditions to re...,if you are a subscriber jagex will treat the f...
4,Privacy Policy,it also enables us to serve you advertising an...,the service uses your personal data to employ ...


In [12]:
# Step 7: Sentence tokenization
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")

from nltk.tokenize import sent_tokenize

# Sentence tokenization
def split_into_sentences(text):
    return [sent.strip() for sent in sent_tokenize(text) if len(sent.strip()) > 5]

df["sentences"] = df["clean_text"].apply(split_into_sentences)

print(df["sentences"].iloc[0][:5])

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


['search encrypt does not track search history in any user identifiable way.', 'the most popular search engines create search profiles of specific users in order to retarget ads based on those search queries as the user navigates the internet.', 'search encrypt does not track your search history in any user identifiable way.']


In [14]:
# Step 8: Chunk long documents by token length
def chunk_sentences_by_tokens(sentences, tokenizer, max_tokens=700, min_sentences=1):
    chunks = []
    current_chunk = []
    current_length = 0

    for sent in sentences:
        sent_tokens = tokenizer.encode(sent, add_special_tokens=False)
        sent_len = len(sent_tokens)

        if sent_len > max_tokens:
            sent_tokens = sent_tokens[:max_tokens]
            sent = tokenizer.decode(sent_tokens, skip_special_tokens=True)
            sent_len = len(sent_tokens)

        if current_length + sent_len <= max_tokens:
            current_chunk.append(sent)
            current_length += sent_len
        else:
            if len(current_chunk) >= min_sentences:
                chunks.append(" ".join(current_chunk))
            current_chunk = [sent]
            current_length = sent_len

    if len(current_chunk) > 0:
        chunks.append(" ".join(current_chunk))

    return chunks

# Chunking for the abstractive / transformer side only
df["chunks"] = df["sentences"].apply(
    lambda sents: chunk_sentences_by_tokens(sents, tokenizer, max_tokens=700)
)

display(df[["doc", "chunks"]].head())

,doc,chunks
0,Privacy Policy,[search encrypt does not track search history ...
1,Privacy Policy,[we also provide you additional data control o...
2,Additional Terms of Service,[rvices you grant oath the following worldwide...
3,Terms and Conditions,[we may change these terms and conditions to r...
4,Privacy Policy,[it also enables us to serve you advertising a...


In [15]:
# Step 9: # Create chunk-level dataset for abstractive summarisation only
expanded_rows = []

for doc_id, row in df.iterrows():
    summary = row["clean_summary"]
    doc_name = row["doc"]

    for chunk_id, chunk in enumerate(row["chunks"]):
        expanded_rows.append({
            "doc_id": doc_id,
            "doc_name": doc_name,
            "chunk_id": chunk_id,
            "input_text": chunk,
            "target_summary": summary
        })

chunk_df = pd.DataFrame(expanded_rows)

print("Chunk-level dataset shape:", chunk_df.shape)
display(chunk_df.head())

Chunk-level dataset shape: (362, 5)


,doc_id,doc_name,chunk_id,input_text,target_summary
0,0,Privacy Policy,0,search encrypt does not track search history i...,this service does not track you.
1,1,Privacy Policy,0,we also provide you additional data control op...,you can request access and deletion of persona...
2,2,Additional Terms of Service,0,rvices you grant oath the following worldwide ...,the copyright license granted to yahoo for pho...
3,3,Terms and Conditions,0,we may change these terms and conditions to re...,if you are a subscriber jagex will treat the f...
4,4,Privacy Policy,0,it also enables us to serve you advertising an...,the service uses your personal data to employ ...


In [16]:
# Step 10: Remove short or invalid rows
chunk_df = chunk_df.dropna().reset_index(drop=True)
chunk_df = chunk_df[chunk_df["input_text"].str.len() > 30]
chunk_df = chunk_df[chunk_df["target_summary"].str.len() > 5]
chunk_df = chunk_df.reset_index(drop=True)

print("Cleaned chunk-level dataset shape:", chunk_df.shape)

Cleaned chunk-level dataset shape: (362, 5)


In [17]:
# Step 11: Split by document id
from sklearn.model_selection import train_test_split

doc_ids = chunk_df["doc_id"].unique()

train_docs, temp_docs = train_test_split(doc_ids, test_size=0.30, random_state=42)
val_docs, test_docs = train_test_split(temp_docs, test_size=0.50, random_state=42)

def assign_split(doc_id):
    if doc_id in train_docs:
        return "train"
    elif doc_id in val_docs:
        return "validation"
    else:
        return "test"

chunk_df["split"] = chunk_df["doc_id"].apply(assign_split)

print(chunk_df["split"].value_counts())
display(chunk_df.head())

split
train         253
test           55
validation     54
Name: count, dtype: int64


,doc_id,doc_name,chunk_id,input_text,target_summary,split
0,0,Privacy Policy,0,search encrypt does not track search history i...,this service does not track you.,train
1,1,Privacy Policy,0,we also provide you additional data control op...,you can request access and deletion of persona...,train
2,2,Additional Terms of Service,0,rvices you grant oath the following worldwide ...,the copyright license granted to yahoo for pho...,train
3,3,Terms and Conditions,0,we may change these terms and conditions to re...,if you are a subscriber jagex will treat the f...,validation
4,4,Privacy Policy,0,it also enables us to serve you advertising an...,the service uses your personal data to employ ...,train


In [18]:
# Step 12: Tokenize for BART / DistilBART
MAX_INPUT_LEN = 768
MAX_TARGET_LEN = 64

def preprocess_for_bart(batch):
    model_inputs = tokenizer(
        batch["input_text"],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=batch["target_summary"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [19]:
# Step 13: Save processed files
chunk_df.to_csv("/content/processed_policy_chunks.csv", index=False)

train_df = chunk_df[chunk_df["split"] == "train"].reset_index(drop=True)
val_df = chunk_df[chunk_df["split"] == "validation"].reset_index(drop=True)
test_df = chunk_df[chunk_df["split"] == "test"].reset_index(drop=True)

train_df.to_csv("/content/train_policy.csv", index=False)
val_df.to_csv("/content/val_policy.csv", index=False)
test_df.to_csv("/content/test_policy.csv", index=False)

print("Saved processed files successfully.")

Saved processed files successfully.


In [20]:
# Step 14: Inspect a few processed examples
for i in range(min(3, len(chunk_df))):
    print("=" * 100)
    print("DOCUMENT TYPE:", chunk_df.loc[i, "doc_name"])
    print("\nINPUT TEXT:\n", chunk_df.loc[i, "input_text"][:700])
    print("\nTARGET SUMMARY:\n", chunk_df.loc[i, "target_summary"])
    print("\nSPLIT:", chunk_df.loc[i, "split"])

DOCUMENT TYPE: Privacy Policy

INPUT TEXT:
 search encrypt does not track search history in any user identifiable way. the most popular search engines create search profiles of specific users in order to retarget ads based on those search queries as the user navigates the internet. search encrypt does not track your search history in any user identifiable way.

TARGET SUMMARY:
 this service does not track you.

SPLIT: train
DOCUMENT TYPE: Privacy Policy

INPUT TEXT:
 we also provide you additional data control options created by the gdpr but provided to the stack overflow community regardless of geographic location with respect to your information including data access and portability including the right to obtain and download a copy of the personal data you provided to stack overflow data correction the ability to update the personal data we collect and display on you in many cases via your account settings data deletion where stack overflow will delete personal information stored on 

In [21]:
print(chunk_df.isnull().sum())

doc_id            0
doc_name          0
chunk_id          0
input_text        0
target_summary    0
split             0
dtype: int64


In [22]:
print("Total rows:", len(chunk_df))
print(chunk_df["split"].value_counts())

Total rows: 362
split
train         253
test           55
validation     54
Name: count, dtype: int64


In [23]:
chunk_df["input_len_chars"] = chunk_df["input_text"].str.len()
chunk_df["summary_len_chars"] = chunk_df["target_summary"].str.len()

print(chunk_df[["input_len_chars", "summary_len_chars"]].describe())

       input_len_chars  summary_len_chars
count       362.000000         362.000000
mean        424.687845          88.812155
std         490.066296          58.825655
min          44.000000          17.000000
25%         171.000000          52.000000
50%         291.000000          77.000000
75%         494.000000         104.000000
max        4108.000000         466.000000


In [24]:
chunk_df["input_len_tokens"] = chunk_df["input_text"].apply(
    lambda x: len(tokenizer.encode(x, add_special_tokens=False))
)
chunk_df["summary_len_tokens"] = chunk_df["target_summary"].apply(
    lambda x: len(tokenizer.encode(x, add_special_tokens=False))
)

print(chunk_df[["input_len_tokens", "summary_len_tokens"]].describe())

       input_len_tokens  summary_len_tokens
count        362.000000          362.000000
mean          76.552486           16.723757
std           85.009947           11.177537
min            8.000000            4.000000
25%           32.000000           10.000000
50%           53.000000           14.000000
75%           91.000000           19.000000
max          696.000000           95.000000


In [25]:
print("Duplicate input rows:", chunk_df.duplicated(subset=["input_text", "target_summary"]).sum())

Duplicate input rows: 0


In [26]:
split_check = chunk_df.groupby("doc_id")["split"].nunique()
print("Docs appearing in multiple splits:", (split_check > 1).sum())

Docs appearing in multiple splits: 0


In [27]:
sample_check = chunk_df.sample(10, random_state=42)

for i, row in sample_check.iterrows():
    print("=" * 100)
    print("DOC TYPE:", row["doc_name"])
    print("INPUT TEXT:\n", row["input_text"][:1000])
    print("\nTARGET SUMMARY:\n", row["target_summary"])
    print("\nSPLIT:", row["split"])

DOC TYPE: Terms of Service
INPUT TEXT:
 the website is available only to individuals who are at least 13 years old.

TARGET SUMMARY:
 this service is only available to users of a certain age.

SPLIT: test
DOC TYPE: Privacy Policy
INPUT TEXT:
 if this policy is substantively updated we will update the text of this page and provide notice to you at duckduckgo com about by writing updated in red next to the link to this page in the footer for a period of at least 30 days.

TARGET SUMMARY:
 conditions may change but your continued acceptance is not inferred from an earlier acceptance flow.

SPLIT: validation
DOC TYPE: Privacy Policy
INPUT TEXT:
 you do not have to give your personal or legal name to create an npm account. you can use a pseudonym instead.

TARGET SUMMARY:
 the service allows you to use pseudonyms.

SPLIT: validation
DOC TYPE: Terms of Use
INPUT TEXT:
 when you submit text to which you hold the copyright you agree to license it under creative commons attribution sharealike 3

In [28]:
# Save full-document version for extractive baseline
extractive_df.to_csv("/content/extractive_full_docs.csv", index=False)

# Save chunk-level version for abstractive / transformer side
chunk_df.to_csv("/content/processed_policy_chunks.csv", index=False)

print("Saved:")
print("- /content/extractive_full_docs.csv")
print("- /content/processed_policy_chunks.csv")

Saved:
- /content/extractive_full_docs.csv
- /content/processed_policy_chunks.csv
